In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
import joblib
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

class MedAssistLocal:
    def __init__(self):
        print("Initializing MedAssist Local Engine...")
        # Define the base path for the data files
        base_path = '/content/drive/MyDrive/sml project/'

        # 1. Load the Model and Tools
        self.model = joblib.load(base_path + 'medassist_model_cpu.joblib')
        self.le = joblib.load(base_path + 'label_encoder.joblib')
        self.matrix = joblib.load(base_path + 'symptom_knowledge_matrix.joblib')
        # Load the saved test data
        self.X_test = np.load(base_path + 'Test data/X_test.npy')
        self.y_test = np.load(base_path + 'Test data/y_test.npy')

        # 2. Load the exact symptom list (order matters!)
        with open(base_path + 'symptom_list.json', 'r') as f:
            self.all_symptoms = json.load(f)

        print(f"System Ready. Loaded {len(self.all_symptoms)} symptoms and {len(self.le.classes_)} diseases.")

    def get_top_3(self, user_inputs):
        # Create input vector (377 zeros)
        input_vector = np.zeros(len(self.all_symptoms))

        # Set 1 for symptoms provided
        for s in user_inputs:
            if s in self.all_symptoms:
                idx = self.all_symptoms.index(s)
                input_vector[idx] = 1

        # Predict Probabilities
        probs = self.model.predict_proba([input_vector])[0]
        top_3_idx = np.argsort(probs)[-3:][::-1]

        results = []
        for idx in top_3_idx:
            disease = self.le.inverse_transform([idx])[0]
            confidence = probs[idx]
            results.append((disease, confidence))
        return results

    def suggest_follow_up(self, user_inputs, n=3):
        # Sum co-occurrence from the knowledge matrix
        scores = self.matrix[user_inputs].sum(axis=1)
        # Remove symptoms already mentioned
        scores = scores.drop(labels=user_inputs)
        return scores.sort_values(ascending=False).head(n).index.tolist()

    def run_Evaluation(self):
      # 1. Get standard Top-1 Predictions
      y_pred = self.model.predict(self.X_test)

      # 2. Get probabilities for Top-3 Accuracy
      probs = self.model.predict_proba(self.X_test)

      # 3. Calculate Top-1 Accuracy
      acc_top1 = accuracy_score(self.y_test, y_pred)

      # 4. Calculate Top-3 Accuracy
      # Check if the true label exists in the indices of the 3 highest probabilities
      top_3_indices = np.argsort(probs, axis=1)[:, -3:]
      correct_in_top3 = np.any(top_3_indices == self.y_test.reshape(-1, 1), axis=1)
      acc_top3 = np.mean(correct_in_top3)

      # 5. Get Macro F1-Score from classification report
      report = classification_report(self.y_test, y_pred, output_dict=True, zero_division=0)
      macro_f1 = report['macro avg']['f1-score']

      # Return as a dictionary for easy access in your simulation
      return {
          "top1_accuracy": acc_top1,
          "top3_accuracy": acc_top3,
          "macro_f1": macro_f1,
          "predictions": y_pred # Keep this in case you need to see what it guessed
      }

RUNNING THE LOCAL SYSTEM

In [25]:
if __name__ == "__main__":
    engine = MedAssistLocal()

    print(engine.run_Evaluation())

    # Example Interaction
    print("\n--- New Patient Session ---")
    initial_symptoms = ['nose deformity', 'sore in nose', 'neck weakness']
    print(f"User reported: {initial_symptoms}")

    # 1. Get Suggestions
    suggestions = engine.suggest_follow_up(initial_symptoms)
    print(f"AI Suggestion: Are you also feeling {suggestions}?")

    # 2. Get Diagnosis
    # Let's assume the user confirmed 'abdominal pain'
    final_input = initial_symptoms + [suggestions[2]]
    predictions = engine.get_top_3(final_input)

    print("\nFinal Differential Diagnosis:")
    for i, (disease, conf) in enumerate(predictions):
        print(f"{i+1}. {disease:<25} | Confidence: {conf*100:.2f}%")

Initializing MedAssist Local Engine...
System Ready. Loaded 377 symptoms and 475 diseases.
{'top1_accuracy': 0.8708262857858798, 'top3_accuracy': np.float64(0.964605650566936), 'macro_f1': 0.8810185751681624, 'predictions': array([320., 363.,  64., ..., 257., 467., 368.], dtype=float32)}

--- New Patient Session ---
User reported: ['nose deformity', 'sore in nose', 'neck weakness']
AI Suggestion: Are you also feeling ['mouth ulcer', 'skin swelling', 'skin lesion']?

Final Differential Diagnosis:
1. impetigo                  | Confidence: 47.71%
2. diabetic peripheral neuropathy | Confidence: 1.16%
3. female genitalia infection | Confidence: 1.15%
